# CMSC 173 &middot; Machine Learning &mdash; Week 7 Lab
## Cross-Validation & Hyperparameter Tuning

One train/validation split is a coin toss &mdash; a lucky split flatters a model, an unlucky one
buries it. **k-fold cross-validation** averages over several splits for a steadier estimate. You
will build k-fold **from scratch**, use it to pick the best `k` for a nearest-neighbours
classifier (graphing the sweep), then get the same answer from scikit-learn in one line.

**How this lab works.** Each part = a short **plain-English explainer**, a **code cell**
you run, a **line-by-line walkthrough** of what it did, and an **Answer here** box. The
code does the maths; we *graph* the results so you can see what is going on.

**NumPy + Matplotlib + scikit-learn.** **Not graded.** About 55 minutes.

---
## Part 0 &middot; Setup + data

A two-class dataset with overlap, so the choice of model actually matters.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV
rng = np.random.default_rng(173)

n = 300
y = rng.integers(0, 2, n)
X = rng.normal(np.c_[np.where(y==1, 1.4, 0.0), np.where(y==1, 1.0, 0.0)], 1.2)
print('data:', X.shape, '| classes:', np.bincount(y))

**Reading the code:** 300 points, two features, two classes that overlap (positives sit a bit
up-and-right of negatives, but the clouds mix). No model will be perfect &mdash; good, because that's
when careful evaluation matters.

---
## Part 1 &middot; Why one split isn't enough

Let's score a model on three *different* random validation splits and watch the number jump
around. That wobble is why we don't trust a single split.

In [ ]:
from sklearn.model_selection import train_test_split
for seed in [0, 1, 2]:
    Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.3, random_state=seed)
    acc = KNeighborsClassifier(7).fit(Xtr, ytr).score(Xva, yva)
    print(f'split seed {seed}: validation accuracy = {acc:.3f}')

**Reading the code:** same data, same model (`k=7` neighbours), only the *split* changes &mdash; and the
accuracy still moves by several points. If you tuned your model against just one of these splits,
you'd partly be tuning to luck.

**Answer here:**

1. The three accuracies differ even though nothing about the model changed. Where does that
   variation come from?
   &rarr; *your answer*

---
## Part 2 &middot; k-fold cross-validation, from scratch

The fix: split the data into `k` equal **folds**. Take turns &mdash; each fold is the validation set
once while the other `k-1` train the model. Average the `k` scores. Every point is used for
validation exactly once, so no single lucky split dominates.

In [ ]:
def kfold_score(model_maker, X, y, k=5):
    idx = rng.permutation(len(y))                     # (1) shuffle the row order
    folds = np.array_split(idx, k)                    # (2) cut into k index groups
    scores = []
    for i in range(k):
        val_idx = folds[i]                            # (3) fold i is validation...
        tr_idx  = np.concatenate([folds[j] for j in range(k) if j != i])  # (4) ...rest is train
        m = model_maker().fit(X[tr_idx], y[tr_idx])   # (5) fresh model each fold
        scores.append(m.score(X[val_idx], y[val_idx]))# (6) score on the held-out fold
    return np.array(scores)

s = kfold_score(lambda: KNeighborsClassifier(7), X, y, k=5)
print('the 5 fold scores :', np.round(s, 3))
print(f'their average     : {s.mean():.3f}   (this is the CV score)')

**Reading the code, line by line:**
- **(1)** shuffle so the folds aren't accidentally ordered.
- **(2)** `np.array_split` cuts the shuffled indices into `k` roughly-equal folds.
- **(3)&ndash;(4)** for each round, one fold is validation and the concatenation of the others is training.
- **(5)** `model_maker()` builds a *fresh* model each fold (never reuse a trained one).
- **(6)** score on the untouched fold. The average of the 5 scores is the **cross-validation score**
  &mdash; steadier than any single split from Part 1.

**Answer here:**

1. The 5 fold scores still vary, but you report their *average*. Why is that average more
   trustworthy than one split's accuracy?
   &rarr; *your answer*

---
## Part 3 &middot; Use CV to pick `k` (the validation curve)

Now the payoff: for a nearest-neighbours classifier, how many neighbours should vote? Too few (`k=1`)
overfits noise; too many blurs the classes. We CV-score every `k` and graph the sweep.

In [ ]:
ks = range(1, 40, 2)
cv_means = [kfold_score(lambda k=k: KNeighborsClassifier(k), X, y, k=5).mean() for k in ks]
best_k = list(ks)[int(np.argmax(cv_means))]

plt.figure(figsize=(7,4))
plt.plot(list(ks), cv_means, 'o-')
plt.axvline(best_k, ls='--', color='gray', label=f'best k = {best_k}')
plt.xlabel('k (number of neighbours)'); plt.ylabel('cross-validation accuracy')
plt.title('Pick the k at the top of the curve'); plt.legend(); plt.tight_layout(); plt.show()
print(f'best k by cross-validation = {best_k}')

**Reading the code:** for each candidate `k` we run 5-fold CV and keep the mean. The curve rises
(leaving behind the jumpy `k=1`), peaks, then sags as `k` gets too large. `argmax` picks the top.
That peak-hunting is **hyperparameter tuning**, done honestly on held-out folds.

**Answer here:**

1. `k=1` scores worse than the peak. In week-2/4 language, is a 1-neighbour model high **bias** or
   high **variance**? What about a very large `k`?
   &rarr; *your answer*

---
## Part 4 &middot; The same thing, in one sklearn line

You built CV to understand it. In practice, `cross_val_score` does exactly this &mdash; compare its
answer to your from-scratch one for `k=7`.

In [ ]:
sk = cross_val_score(KNeighborsClassifier(7), X, y, cv=5)   # sklearn's 5-fold CV
print('sklearn fold scores:', np.round(sk, 3))
print(f'sklearn CV mean    : {sk.mean():.3f}')
print(f'(your from-scratch : {kfold_score(lambda: KNeighborsClassifier(7), X, y, 5).mean():.3f})')

**Reading the code:** `cross_val_score(model, X, y, cv=5)` returns the five fold scores in one call.
The mean is essentially the same as yours (small differences come only from how the folds are
shuffled). Now you know what that one line is *doing*.

---
## Part 5 &middot; Grid search: tune automatically

When you have several hyperparameters, you don't sweep by hand &mdash; `GridSearchCV` tries every
combination with CV and hands back the winner.

In [ ]:
grid = GridSearchCV(
    KNeighborsClassifier(),
    {'n_neighbors': list(range(1, 40, 2)), 'weights': ['uniform', 'distance']},  # the grid
    cv=5)
grid.fit(X, y)
print('best settings :', grid.best_params_)
print(f'best CV score : {grid.best_score_:.3f}')

**Reading the code:** we hand `GridSearchCV` a model and a dictionary of options (here: every odd `k`
and two voting schemes). It runs 5-fold CV for *every* combination and stores the best in
`best_params_`. It's Part 3 automated across more than one knob.

**Answer here:**

1. Grid search tried ~40 combinations &times; 5 folds = ~200 fits for this tiny grid. Why can grid
   search get *expensive* fast, and what's one cheaper alternative from lecture?
   &rarr; *your answer*

---
## Where you actually are

Set the pace honestly. Replace each `-` with: **solid** / **rusty** / **never really got it**.

| | You |
|---|---|
| Why one split is unreliable | - |
| How k-fold works (in words) | - |
| Reading a validation curve | - |
| Using cross_val_score | - |
| What GridSearchCV automates | - |

**Which part took longest, and where did you get stuck?**
&rarr; *your answer*

**In one plain sentence: what does cross-validation buy you over a single train/val split?**
&rarr; *your answer*

---
## Stretch &mdash; optional

Required part is done; nothing below is graded.

### Stretch &middot; Try 10-fold

Re-run the `k=7` CV with `k=10` folds instead of 5. Does the average change much? Fill the line.

In [ ]:
# your code here: print kfold_score(lambda: KNeighborsClassifier(7), X, y, k=10).mean()


---
## Submitting

Run the cell below. It uploads this notebook straight from Colab &mdash; nothing to download.

You need a **submit token**: open
[https://portal.latarak.com/student/submit-token](https://portal.latarak.com/student/submit-token),
sign in, press the button, then paste it when the cell asks. The cell hides what you type.

In [ ]:
# --- Submit this notebook ------------------------------------------------------
# Colab only. Anywhere else, use the manual route described below this cell.
import getpass, json, urllib.request, urllib.error

PORTAL, COURSE, WEEK = "https://portal.latarak.com", "cmsc173", 7

try:
    from google.colab import _message
except ImportError:
    raise SystemExit(
        "Not running in Colab. Download this notebook "
        "(File > Download > Download .ipynb) and upload it at "
        "https://portal.latarak.com/course/cmsc173/lab/7/submit"
    )

nb = _message.blocking_request("get_ipynb", timeout_sec=90)["ipynb"]
token = getpass.getpass("Submit token (hidden as you type): ").strip()

req = urllib.request.Request(
    PORTAL + "/api/labs/" + COURSE + "/submit-notebook",
    data=json.dumps({"week": WEEK, "notebook": nb}).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer " + token},
    method="POST",
)
try:
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.load(r)
    print("Submitted", out["course"], "week", out["week"], "for", out["student"])
    print(out["cells"], "cells,", out["executed"], "executed")
    print(out["message"])
except urllib.error.HTTPError as e:
    print("Not submitted:", json.loads(e.read()).get("error", e.reason))

Prefer to do it by hand? **File &rarr; Download &rarr; Download .ipynb**, then go to the
[Week 7 submission page](https://portal.latarak.com/course/cmsc173/lab/7/submit) and upload it.

Blank cells are fine and guesses are fine. Don't polish this until it hides what you knew.